# Document RAG analysis with open source LLMs

In [9]:
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter

file_path = './files/ipaybtc-faq.txt'
# Load Document
with open(file_path) as f:
    doc = f.read()


# Split Document
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 150,
    chunk_overlap=20,
    length_function=len
)    

# Create chunks
chunks = text_splitter.split_text(doc)

print(f"Chunks: {len(chunks)}")

for i, chunk in enumerate(chunks):
    print(f"Chunk {i} - {chunk} \n")

Chunks: 451
Chunk 0 - Our Vision 

Chunk 1 - ipayBTC has a bold vision of building a world where satoshi is the dominant unit of account, a world where the majority of people earn, save and 

Chunk 2 - earn, save and spend their value in satoshi. 

Chunk 3 - Our Mission
ipayBTC aims to incorporate satoshi payments into all facets of our everyday life. 

Chunk 4 - Cross-border Payments
With ipayBTC, users can receive instant and inexpensive cross-border payments using the Lightning Network. 

Chunk 5 - Micro Payments
ipayBTC supports the Lightning Network, allowing small businesses and freelancers to handle small and micro-transactions efficiently. 

Chunk 6 - Financial Inclusion
ipayBTC addresses financial inclusion by educating users through various content formats on every platform it is present on. 

Chunk 7 - Which ID verification type should I use? 

Chunk 8 - To ensure security and smooth customer experience, we require customers to verify their identity. For ID verification, you

In [10]:
# Embeddings
from sentence_transformers import SentenceTransformer

# Load embedding model
sentenceModel = SentenceTransformer('all-MiniLM-L6-v2')

chunk_emdeddings = sentenceModel.encode(chunks)

print(f"Shape of embeddings: {chunk_emdeddings.shape}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2091.68it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Shape of embeddings: (451, 384)


In [11]:
# Vector store
import faiss
import numpy as np

# Get dimension
d = chunk_emdeddings.shape[1]

# Create a FAISS index
index = faiss.IndexFlatL2(d)

# Add chunk embeddings to the index
index.add(np.array(chunk_emdeddings).astype('float32'))

print(f"FAISS index created with {index.ntotal} vectors.")

FAISS index created with 451 vectors.


In [12]:
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSeq2SeqLM
import os
import torch

# Load summarization pipeline with a smaller model

token = os.getenv("HF_TOKEN")

device = "mps" if torch.backends.mps.is_available() else "cpu"


 # google/flan-t5-base,  google/flan-t5-large, microsoft/phi-3-mini, Qwen2.5-0.5B, TinyLlama-1.1B, meta-llama/Llama-3.2-1B-Instruct
# model_name = "google/gemma-2b-it"
model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name, token=token)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name, token=token).to(device)

Loading weights: 100%|██████████| 282/282 [00:00<00:00, 2353.50it/s, Materializing param=shared.weight]                                                       
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [13]:
def generate_answer(prompt):
    
    inputs = tokenizer(prompt, return_tensors="pt", max_length=1024, truncation=True)

    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.inference_mode():
        outputs = model.generate(**inputs,          
                                    max_new_tokens=150, 
                                    length_penalty=2.0, 
                                    num_beams=1,
                                    early_stopping=True,)
        
        result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return result

In [ ]:
from transformers import pipeline


def answer_question(query):
    query_embedding = sentenceModel.encode([query]).astype('float32')
    
    # Search the FAISS index for the top k
    k = 2
    distances, indices = index.search(query_embedding, 5)
    
    # Get the text chunks from chunks list
    retrieved_chunks = [chunks[i] for i in indices[0]]
    context = "\n\n".join(retrieved_chunks)
    
    prompt = f"""    
        You are Debbie, a friendly customer support assistant for iPayBTC.
        Speak in a warm, conversational tone like a real human support agent.

        Rules:
        - This user name is Jesse
        - If the user sends a greeting (e.g. "Hello", "Hi", "Hey"), respond warmly and ask how you can help
        - Answer ONLY using the provided context for all other questions
        - If the context doesn't contain the answer, say: "I'm sorry, I don't have that information. Please contact our support team for help! 😊"
        - DO NOT generate answers from your own knowledge
        - Use simple, friendly language
        - Use occasional emojis where appropriate 😊
        - Only greet if the message is JUST a greeting with no question
        - If the user asks a question, answer directly without greeting
    Context:
    {context}

    Customer issue: {query}
    """
    
    answer = generate_answer(prompt)
    
    # print(f"""
    #     Prompt: \n
    #     {prompt} \n
    
    #     Answer: \n
    #     {answer}
    # """)
    
    return answer.strip()


In [22]:
query = "How do I become a buy USDT"  # Test greeting

print(f"Answer: {answer_question(query)}")

Answer: You can buy USDT from any of the major exchanges.
